In [1]:
# ===== Imports (ครบทุกตัวที่ใช้) =====
from pathlib import Path
from collections import defaultdict
import random
import shutil
import csv

import cv2

# ===== Config =====
INPUT_ROOT = Path("NEW DATA")     # มีโฟลเดอร์ FAKE/REAL (วิดีโอ)
FAKE_DIR = INPUT_ROOT / "FAKE"
REAL_DIR = INPUT_ROOT / "REAL"

EXPORT_FRAMES = Path("Export/paired_frames")  # เฟรมที่แตกแล้ว (ก่อน split)
DATA_ROOT = Path("data")                      # train/val จะอยู่ในนี้
META_CSV = DATA_ROOT / "metadata.csv"

VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv", ".webm", ".m4v"}

SEED = 42

# Timestamp extraction
START_SEC = 1   # เริ่มที่วินาทีที่ 1
STEP_SEC = 1    # ทุกๆ 1 วินาที

# Split
TRAIN_RATIO = 0.8  # 80% train, 20% val

print("Config loaded.")
print("FAKE_DIR:", FAKE_DIR)
print("REAL_DIR:", REAL_DIR)


Config loaded.
FAKE_DIR: NEW DATA/FAKE
REAL_DIR: NEW DATA/REAL


In [2]:
def extract_core_from_video_name(filename):
    """
    FAKE: 01_02__exit_phone_room__YVGY8LOK.mp4 -> exit_phone_room
    REAL: 01__exit_phone_room.mp4              -> exit_phone_room
    """
    name = Path(filename).stem
    parts = name.split("__")
    if len(parts) >= 2:
        return parts[1]
    return None

def index_videos_by_core(folder):
    idx = defaultdict(list)
    for p in Path(folder).rglob("*"):
        if p.is_file() and p.suffix.lower() in VIDEO_EXTS:
            core = extract_core_from_video_name(p.name)
            if core:
                idx[core].append(p)
    return dict(idx)

def get_one_to_one_pairs(fake_index, real_index, seed=42):
    rng = random.Random(seed)
    common_cores = sorted(set(fake_index) & set(real_index))
    pairs = []
    for core in common_cores:
        f = rng.choice(fake_index[core])
        r = rng.choice(real_index[core])
        pairs.append((core, f, r))
    return pairs

def get_duration_seconds(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        return 0.0
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()
    if fps and fps > 0 and frame_count and frame_count > 0:
        return float(frame_count) / float(fps)
    return 0.0

def read_frame_at_second(video_path, sec):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        return None
    cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000)
    ok, frame = cap.read()
    cap.release()
    if not ok or frame is None:
        return None
    return frame

def extract_paired_frames_by_second(pairs, out_root, start_sec=1, step_sec=1):
    out_root = Path(out_root)
    (out_root / "FAKE").mkdir(parents=True, exist_ok=True)
    (out_root / "REAL").mkdir(parents=True, exist_ok=True)

    total_saved = 0

    for core, fake_path, real_path in pairs:
        dur_f = get_duration_seconds(fake_path)
        dur_r = get_duration_seconds(real_path)
        max_sec = int(min(dur_f, dur_r))  # floor

        if max_sec < start_sec:
            print(f"[SKIP] {core}: too short (max_sec={max_sec})")
            continue

        saved = 0
        for sec in range(start_sec, max_sec + 1, step_sec):
            f_frame = read_frame_at_second(fake_path, sec)
            r_frame = read_frame_at_second(real_path, sec)
            if f_frame is None or r_frame is None:
                continue

            f_out = out_root / "FAKE" / f"{core}__t{sec:04d}__fake.jpg"
            r_out = out_root / "REAL" / f"{core}__t{sec:04d}__real.jpg"

            cv2.imwrite(str(f_out), f_frame)
            cv2.imwrite(str(r_out), r_frame)

            saved += 1
            total_saved += 2

        print(f"[OK] {core}: saved {saved} paired seconds")

    print("Total frames written (fake+real):", total_saved)


In [3]:
def extract_core_from_frame_name(filename):
    # core__t0001__fake.jpg -> core
    return filename.split("__t")[0]

def index_frames_by_core(frames_root, cls_folder):
    idx = defaultdict(list)
    frames_root = Path(frames_root)
    for img in (frames_root / cls_folder).glob("*.jpg"):
        core = extract_core_from_frame_name(img.name)
        idx[core].append(img)
    return dict(idx)

def split_cores(cores, train_ratio=0.8, seed=42):
    rng = random.Random(seed)
    cores = list(cores)
    rng.shuffle(cores)
    n_train = int(len(cores) * train_ratio)
    return cores[:n_train], cores[n_train:]

def copy_split(frames_index, cores, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    count = 0
    for core in cores:
        for p in frames_index.get(core, []):
            shutil.copy(p, out_dir / p.name)
            count += 1
    return count

def infer_sec_from_name(name):
    # core__t0007__fake.jpg -> 7
    try:
        t_part = name.split("__t")[1]
        sec_str = t_part.split("__")[0]
        return int(sec_str)
    except:
        return None


In [4]:
# ---- Safety checks ----
if not FAKE_DIR.exists():
    raise FileNotFoundError(f"Not found: {FAKE_DIR}")
if not REAL_DIR.exists():
    raise FileNotFoundError(f"Not found: {REAL_DIR}")

# (optional) ล้าง output เก่า ถ้าคุณอยากเริ่มใหม่สะอาด ๆ
# shutil.rmtree(EXPORT_FRAMES, ignore_errors=True)
# shutil.rmtree(DATA_ROOT, ignore_errors=True)

# 1) Index videos
fake_index = index_videos_by_core(FAKE_DIR)
real_index = index_videos_by_core(REAL_DIR)

common_cores = sorted(set(fake_index) & set(real_index))
print("Matched scenarios (cores):", len(common_cores))
print("Example cores:", common_cores[:10])

# 2) Pair 1:1 per core
pairs = get_one_to_one_pairs(fake_index, real_index, seed=SEED)
print("Total pairs:", len(pairs))

# 3) Extract timestamp-synced frames
EXPORT_FRAMES.mkdir(parents=True, exist_ok=True)
extract_paired_frames_by_second(pairs, EXPORT_FRAMES, start_sec=START_SEC, step_sec=STEP_SEC)

# 4) Index frames by core
fake_frames = index_frames_by_core(EXPORT_FRAMES, "FAKE")
real_frames = index_frames_by_core(EXPORT_FRAMES, "REAL")
frame_common_cores = sorted(set(fake_frames) & set(real_frames))
print("Cores with extracted frames:", len(frame_common_cores))

# 5) Split cores train/val (core-aware)
train_cores, val_cores = split_cores(frame_common_cores, train_ratio=TRAIN_RATIO, seed=SEED)
print("Train cores:", len(train_cores), "| Val cores:", len(val_cores))

# 6) Copy to data/train|val/(fake|real)
train_fake_dir = DATA_ROOT / "train" / "fake"
train_real_dir = DATA_ROOT / "train" / "real"
val_fake_dir   = DATA_ROOT / "val"   / "fake"
val_real_dir   = DATA_ROOT / "val"   / "real"

n_tf = copy_split(fake_frames, train_cores, train_fake_dir)
n_tr = copy_split(real_frames, train_cores, train_real_dir)
n_vf = copy_split(fake_frames, val_cores,   val_fake_dir)
n_vr = copy_split(real_frames, val_cores,   val_real_dir)

print("Copied:")
print(" train fake:", n_tf, " train real:", n_tr)
print(" val   fake:", n_vf, " val   real:", n_vr)

# 7) Write metadata.csv
DATA_ROOT.mkdir(parents=True, exist_ok=True)
rows = []
for split_name in ["train", "val"]:
    for label in ["fake", "real"]:
        for img in (DATA_ROOT / split_name / label).glob("*.jpg"):
            rows.append({
                "split": split_name,
                "label": label,
                "core": extract_core_from_frame_name(img.name),
                "sec": infer_sec_from_name(img.name),
                "path": str(img)
            })

with open(META_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["split", "label", "core", "sec", "path"])
    w.writeheader()
    w.writerows(rows)

print("Saved metadata:", META_CSV, "| rows:", len(rows))
print("DONE ✅")


Matched scenarios (cores): 16
Example cores: ['exit_phone_room', 'hugging_happy', 'kitchen_pan', 'kitchen_still', 'meeting_serious', 'outside_talking_pan_laughing', 'outside_talking_still_laughing', 'podium_speech_happy', 'secret_conversation', 'talking_against_wall']
Total pairs: 16
[OK] exit_phone_room: saved 16 paired seconds
[OK] hugging_happy: saved 32 paired seconds
[OK] kitchen_pan: saved 30 paired seconds
[OK] kitchen_still: saved 35 paired seconds
[OK] meeting_serious: saved 37 paired seconds
[OK] outside_talking_pan_laughing: saved 29 paired seconds
[OK] outside_talking_still_laughing: saved 27 paired seconds
[OK] podium_speech_happy: saved 34 paired seconds
[OK] secret_conversation: saved 1 paired seconds
[OK] talking_against_wall: saved 37 paired seconds
[OK] talking_angry_couch: saved 63 paired seconds
[OK] walk_down_hall_angry: saved 17 paired seconds
[OK] walking_and_outside_surprised: saved 47 paired seconds
[OK] walking_down_indoor_hall_disgust: saved 34 paired seconds

In [ ]:
read_fail = 0
no_face = 0
used = 0
total = 0

for image_name in dirs:
    total += 1
    image_path = os.path.join(data_folder_path, image_name)

    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print("[read_fail]", image_path)
        read_fail += 1
        continue

In [ ]:
img_bgr = cv2.imread(image_path)
if img_bgr is None:
    print("[read_fail]", image_path)
    read_fail += 1
    continue